<a href="https://colab.research.google.com/github/ford442/the_jokesters/blob/main/utils/convert_kimi_vl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vicuna-7B FP32 ONNX Export (+ Kimi-VL reality check)

**Preferred path (2026):** use the dedicated Docker HF Space instead of Colab:

→ **`hf_spaces/kimi-vl-converter/`** — Gradio **ZeroGPU** Space, pinned `transformers==4.51.3` + `optimum-onnx` (no Docker; ZeroGPU is Gradio-only).

Deploy: create a Space with SDK **Gradio**, hardware **ZeroGPU**, secret `HF_TOKEN` for Hub upload. See that folder's README.

**This notebook** remains as a Colab fallback if you already have a stable A100 session.

**Purpose**: Export Vicuna-7B to FP32 ONNX for WebGPU / ORT Web. Document why **stock Optimum cannot** export `moonshotai/Kimi-VL-A3B-*`.

**Runtime**: A100 (40GB) Colab recommended for Vicuna FP32 — or the converter Space GPU.

### Why Colab cells keep failing

| Symptom | Cause |
|---|---|
| `optimum … does not provide the extra 'exporters'` | Optimum **2.x** moved ONNX export to **`optimum-onnx`**. Use `optimum[onnx]` / `optimum-onnx[onnxruntime-gpu]`, not `optimum[exporters]`. |
| `gradio … requires huggingface-hub<2.0,>=1.2` vs `huggingface-hub 0.36.2` | Colab preinstall vs pinned `transformers==4.51.x`. Prefer the **Docker Space** (no Gradio conflict). |
| `No module named 'transformers.masking_utils'` | Diffusers / Transformers version skew on Colab. Do **not** need Diffusers for this export — uninstall or use the Space. |
| `Unrecognized model … Kimi-VL-A3B-Thinking-2506` | Config `model_type` is custom `kimi_vl` (remote code). Even with `trust_remote_code=True`, Optimum has **no** built-in ONNX config for MoonViT + DeepSeek-style MoE VL. |

### Product path in The Jokesters

Kimi-VL is registered as a **server API** model (`src/config/models.ts`), not a browser ONNX/WebLLM weight. Multimodal Kimi is an experimental track (see `docs/adr/0001-native-cpp-boundary.md`). Prefer `llama-server` / ZeroGPU Space over browser ONNX.


In [ ]:
# CELL 1 — Clean env (A100 recommended)
# After this cell finishes: Runtime → Restart session, then run CELL 1b onward.

!pip uninstall -y optimum optimum-onnx onnxruntime onnxruntime-gpu 2>/dev/null || true

# Pin the Kimi-VL / Jokesters-verified transformers line.
# ONNX export lives in optimum-onnx (Optimum 2.x); do NOT use optimum[exporters].
!pip install -q --upgrade pip
!pip install -q \
  "transformers==4.51.3" \
  "accelerate>=0.27.0,<1.0" \
  "huggingface_hub>=0.30.0,<1.0" \
  "optimum-onnx[onnxruntime-gpu]==0.1.0" \
  "onnx" \
  "safetensors" \
  "sentencepiece" \
  "protobuf"

# Optional: stop Diffusers from breaking imports on Colab (masking_utils / Flax noise).
!pip uninstall -y diffusers 2>/dev/null || true

print("✅ Packages installed.")
print("⚠️  Restart the runtime NOW (Runtime → Restart session), then run CELL 1b.")

In [ ]:
# CELL 1b — Verify pins (run AFTER runtime restart)
import os
import transformers
import huggingface_hub

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("transformers", transformers.__version__)
print("huggingface_hub", huggingface_hub.__version__)

assert transformers.__version__.startswith("4.51"), (
    f"Need transformers 4.51.x for Kimi-VL remote code; got {transformers.__version__}. "
    "Restart runtime after CELL 1."
)

from optimum.exporters.onnx import main_export  # noqa: F401
print("✅ optimum.exporters.onnx import OK (via optimum-onnx)")

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# CELL 2 — Preflight: can we load Kimi-VL config with trust_remote_code?
from transformers import AutoConfig, AutoProcessor

KIMI_ID = "moonshotai/Kimi-VL-A3B-Thinking-2506"  # or …/Kimi-VL-A3B-Instruct

cfg = AutoConfig.from_pretrained(KIMI_ID, trust_remote_code=True)
print("model_type:", cfg.model_type)
print("architectures:", getattr(cfg, "architectures", None))
assert cfg.model_type == "kimi_vl", cfg.model_type

# Processor only — do not load full 16B MoE weights unless you intend to.
proc = AutoProcessor.from_pretrained(KIMI_ID, trust_remote_code=True)
print("✅ Remote code + processor OK:", type(proc).__name__)
print(
    "Note: Optimum TasksManager has no onnx config for model_type='kimi_vl'. "
    "main_export without custom_onnx_configs will fail after this point."
)

In [ ]:
# CELL 3 — Export Vicuna-7B → FP32 ONNX (supported path)
!optimum-cli export onnx \
  --model lmsys/vicuna-7b-v1.5 \
  --task text-generation-with-past \
  --device cuda \
  --dtype fp32 \
  --optimize O2 \
  --output /content/vicuna_7b_onnx_fp32

print("✅ Vicuna 7B FP32 exported → /content/vicuna_7b_onnx_fp32")
!ls -lh /content/vicuna_7b_onnx_fp32 | head

In [ ]:
# CELL 4 — Kimi-VL ONNX: expected failure + supported alternatives
#
# Stock Optimum cannot export kimi_vl (custom MoonViT + MoE decoder).
# Old notebook kwargs that break on optimum-onnx 0.1.x:
#   no_fp16=True              → use dtype="fp32"
#   use_external_data_format  → removed / handled by exporter
#
from pathlib import Path
from optimum.exporters.onnx import main_export

KIMI_ID = "moonshotai/Kimi-VL-A3B-Thinking-2506"
OUT = Path("/content/kimi_vl_onnx_fp32")
OUT.mkdir(parents=True, exist_ok=True)

TRY_EXPORT = False  # set True only to capture the Optimum error for debugging

if TRY_EXPORT:
    try:
        main_export(
            model_name_or_path=KIMI_ID,
            output=str(OUT),
            task="image-text-to-text",  # closest VL task synonym; still unsupported for kimi_vl
            trust_remote_code=True,
            device="cuda",
            dtype="fp32",
            optimize="O2",
            opset=17,
            batch_size=1,
            # custom_onnx_configs=...  # required for custom arches; not provided upstream for kimi_vl
        )
        print("✅ Unexpected success — inspect", OUT)
    except Exception as exc:
        print("❌ Expected Optimum failure for kimi_vl:")
        print(type(exc).__name__ + ":", exc)
else:
    print("⏭️  Skipping Kimi-VL Optimum export (TRY_EXPORT=False).")

print(
    """
Supported alternatives for The Jokesters:
  0. Converter Space (preferred) — hf_spaces/kimi-vl-converter/ (Gradio ZeroGPU, pinned deps; Vicuna ONNX + Kimi preflight).
  1. Server API — models.ts entry Kimi-VL-A3B-Thinking-2506 via llama-server / OpenAI-compatible proxy
     (backend/llama_proxy.py; mmproj for vision).
  2. ZeroGPU benchmark Space — hf_spaces/kimi-vl-zero-gpu-test/ (PyTorch, not ONNX).
  3. Research-only — hand-written custom_onnx_configs + MoE-safe graph (not productized).
"""
)


In [ ]:
# CELL 5 — Convert Vicuna ONNX to web-friendly external data shards
import os
import onnx

def convert_to_web_format(input_path, output_dir, shard_mb=512):
    """Rewrite ONNX with external weight files (browser-friendly)."""
    os.makedirs(output_dir, exist_ok=True)
    model = onnx.load(input_path)

    for tensor in model.graph.initializer:
        if tensor.data_type == 10:  # FLOAT16
            print(f"Warning: {tensor.name} is FP16 — prefer re-export with dtype=fp32")

    out_path = f"{output_dir}/model.onnx"
    onnx.save_model(
        model,
        out_path,
        save_as_external_data=True,
        location="weights",
        size_threshold=shard_mb * 1024 * 1024,
        convert_attribute=True,
    )
    shards = [f for f in os.listdir(output_dir) if f.startswith("weights")]
    print(f"✅ {out_path} (+ {len(shards)} weight shard file(s))")
    return out_path

convert_to_web_format("/content/vicuna_7b_onnx_fp32/model.onnx", "/content/web_vicuna")

# Only if CELL 4 somehow produced a graph:
kimi_onnx = "/content/kimi_vl_onnx_fp32/model.onnx"
if os.path.isfile(kimi_onnx):
    convert_to_web_format(kimi_onnx, "/content/web_kimi")
else:
    print("⏭️  No Kimi ONNX to shard (expected).")

In [ ]:
# CELL 6 — Upload Vicuna (optional; requires HF login)
# !huggingface-cli login

from huggingface_hub import HfApi

UPLOAD = False  # set True after `huggingface-cli login`
api = HfApi()

if UPLOAD:
    print("⬆️ Uploading Vicuna…")
    api.upload_folder(
        folder_path="/content/web_vicuna",
        repo_id="ford442/vicuna-7b-webgpu",
        repo_type="model",
    )
    print("✅ Vicuna uploaded")
else:
    print("⏭️  UPLOAD=False — set True to push /content/web_vicuna")